# Advanced packages - MVR (Water Mover)

Part of the **synthetic-valley advanced-packages** series, which upgrades a calibrated base model's simple boundary conditions to MODFLOW 6 *advanced packages* one notebook at a time. Each notebook loads the shared advanced model from `models/` (creating it from the base model on first use with `mf6_notebook_helpers.load_or_create_advanced_model`) and adds one package, so they can be run independently and in any order - except where a package depends on others.

- **UZF** (`mf6-adv-uzf`) - recharge (RCH) -> Unsaturated Zone Flow
- **MAW** (`mf6-adv-maw`) - wells (WEL) -> Multi-Aquifer Well
- **LAK** (`mf6-adv-lak`) - high-K lake -> Lake package
- **SFR** (`mf6-adv-sfr`) - river (RIV) -> Streamflow Routing
- **MVR** (`mf6-adv-mvr`) - Water Mover: runoff and seepage to the lake and stream (requires UZF, LAK, SFR) *(this notebook)*
- **Processing output** (`mf6-adv-processing`) - run the model and plot results

Add the Water Mover (MVR) package to route water between the advanced packages - here sending a fraction of each UZF cell's rejected infiltration to the lake or the stream network.

> **Dependencies.** The mover connects the UZF, LAK, and SFR packages, so all three must already exist in the advanced model. This notebook calls `mf6_notebook_helpers.require_packages` at the start and stops with an explanatory error if any are missing - run the UZF, LAK, and SFR notebooks first.

## What this notebook covers

Connect the advanced packages to each other with the Water Mover (**MVR**)
package, which routes water a package would otherwise reject or discharge into
another package instead of out of the model.

By the end of this notebook you will be able to:

- say which packages can provide and receive water through the mover,
- route a fraction of each UZF cell's available water to the lake or the stream,
- check that the packages MVR depends on are already built, and
- add the mover to the shared model and run it.


## Imports and setup

Import FloPy and the helpers, then choose the temporal resolution
(`sample_frequency`) and the model name.

In [ ]:
import flopy
from mf6_notebook_helpers import (
    drop_packages,
    load_land_surface,
    load_or_create_advanced_model,
    require_packages,
)

In [ ]:
# exercise: choose the temporal resolution and set the model name
sample_frequency = "annual"  # monthly or annual
name = "sv"

## Load the advanced model and check dependencies

Load the shared advanced model and require that UZF, LAK, and SFR have already been built; the seepage drains (DRN) come with the base model. If any is missing, `require_packages` raises with a message naming the notebook to run first.

In [ ]:
# exercise: load the shared advanced model and check the packages the mover needs are present
sim = load_or_create_advanced_model(sample_frequency, name)
gwf = sim.get_model(name)

require_packages(gwf, ["UZF-1", "LAK-1", "SFR-1", "DRN-1"], "Water Mover (MVR)")
uzf = gwf.get_package("UZF-1")
drn = gwf.get_package("DRN-1")

## Build the MVR package

Water that cannot soak in runs off over the land surface to the lake or to the stream reach the land drains to, and so does groundwater the seepage drains remove where the water table reaches the land surface. `runoff_to` in the land surface data gives the destination of each cell: a stream reach (numbered from 0) or -1 for the lake. The mover sends 10 percent of each unsaturated zone cell's runoff and all of each drain's water there.

In [ ]:
surface = load_land_surface()
runoff_to = surface["runoff_to"].to_numpy()


def receiver(cellid):
    """The package and number of the lake or stream reach a cell's water goes to."""
    to = int(runoff_to[cellid[1], cellid[2]])
    return ("LAK-1", 0) if to < 0 else ("SFR-1", to)


# exercise: send each cell's runoff and each drain's water to where the land drains
packages = [["UZF-1"], ["LAK-1"], ["SFR-1"], ["DRN-1"]]
perioddata = []
for ifno, cellid in zip(
    uzf.packagedata.array["ifno"], uzf.packagedata.array["cellid"], strict=True
):
    perioddata.append(("UZF-1", int(ifno), *receiver(cellid), "factor", 0.1))
for n, cellid in enumerate(drn.stress_period_data.get_data(0)["cellid"]):
    perioddata.append(("DRN-1", n, *receiver(cellid), "factor", 1.0))
# end exercise
# the drains pass their water to the mover
drn.mover = True

In [ ]:
# add the mover (idempotent: drop any prior MVR first)
drop_packages(gwf, "mvr", "MVR")
# exercise: create the water mover package
mvr = flopy.mf6.ModflowGwfmvr(
    gwf,
    maxmvr=len(perioddata),
    maxpackages=len(packages),
    packages=packages,
    perioddata={0: perioddata},
)
# end exercise
# a tighter linear solver closure lets the steady state converge: the water the
# mover sends to the short stream reaches then stops changing between iterations
sim.get_package("ims").inner_dvclose = 3.048e-9
gwf.get_package_list()

## Write and run

Write the updated advanced model back to `models/` and run it to confirm the mover-coupled model is valid.

In [ ]:
# exercise: write the input files and run the model
sim.write_simulation(silent=True)
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
# end exercise
print("MVR advanced model ran successfully")

## Recap

In this notebook you added the Water Mover (**MVR**) package, which sends part of each unsaturated zone (UZF) cell's runoff and all of the seepage drains' water to the lake (LAK) or the stream reach (SFR) the land surface drains to, then wrote the updated advanced model back to `models/` and ran it. Because MVR connects UZF, LAK, SFR, and the drains, those packages must already be built.